
# Jour 3 : Les algorithmes du ML et le Fight des IA

## Phase 0: Mise en route

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    r2_score, mean_absolute_error, root_mean_squared_error,
    f1_score
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

## Phase A : Prédire les prix immobiliers (régression)

In [2]:
from sklearn.datasets import fetch_california_housing

def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Doit afficher : nombre de lignes, nombre de variables, et l'unité de la cible.
    """
    data = fetch_california_housing()
    X, y = data.data, data.target

    print(f"California Housing : {data.data.shape}")
    print(f"Variables : {list(data.feature_names)}")
    print(f"Cible : prix médian en centaines de milliers de $")

    return X, y

In [3]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}."""
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    return {
        "r2"  : r2_score(y_test, y_pred),
        "mae" : mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# === CAS NORMAL ===
print("=== CAS NORMAL ===\n")
X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    res = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} : R2={res['r2']:.2f}  MAE={res['mae']:.2f}  RMSE={res['rmse']:.2f}")

# === CAS LIMITE : 100 lignes seulement ===
print("\n=== CAS LIMITE (100 lignes) ===\n")
X_s, y_s = X[:100], y[:100]
Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
Xtr_s = StandardScaler().fit_transform(Xtr)
Xte_s = StandardScaler().fit(Xtr).transform(Xte)

res_petit = evaluer_regression(LinearRegression(), Xtr_s, Xte_s, ytr, yte)
print(f"LinearRegression (100 lignes) : R2={res_petit['r2']:.2f}  MAE={res_petit['mae']:.2f}")
print("→ R2 s'effondre : avec peu de données, le modèle n'a pas assez d'exemples pour généraliser.")

# === CAS ADVERSARIAL : quartier fictif hors plage ===
print("\n=== CAS ADVERSARIAL (quartier fictif) ===\n")
# Revenu médian = 0, 9000 habitants, reste à la moyenne du dataset
quartier_fictif = np.zeros((1, X.shape[1]))
quartier_fictif[0, 0] = 0       # MedInc = 0 (hors plage, min réel ~0.5)
quartier_fictif[0, 5] = 9000    # Population = 9000 (très élevé)

# On réentraîne un modèle propre sur tout le train
lr = LinearRegression()
lr.fit(X_train_s, y_train)

fictif_scaled = scaler.transform(quartier_fictif)
pred = lr.predict(fictif_scaled)[0]
print(f"Prédiction sur quartier fictif : {pred:.2f} (×100k$)")
if pred < 0:
    print("Valeur négative ! Le linéaire extrapole sans garde-fou.")
else:
    print(f"Valeur potentiellement aberrante ({pred:.2f}) hors de la plage réelle [0.15, 5.0]")
print("→ En production : valider les entrées (clipping, rejection) avant de prédire.")

=== CAS NORMAL ===

California Housing : (20640, 8)
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible : prix médian en centaines de milliers de $
LinearRegression     : R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         : R2=0.81  MAE=0.33  RMSE=0.51

=== CAS LIMITE (100 lignes) ===

LinearRegression (100 lignes) : R2=0.71  MAE=0.38
→ R2 s'effondre : avec peu de données, le modèle n'a pas assez d'exemples pour généraliser.

=== CAS ADVERSARIAL (quartier fictif) ===

Prédiction sur quartier fictif : -68.76 (×100k$)
Valeur négative ! Le linéaire extrapole sans garde-fou.
→ En production : valider les entrées (clipping, rejection) avant de prédire.


## Phase B: Segmenter les clients AirBnB (clustering)

In [15]:
def charger_airbnb(chemin):
    """Charge le CSV Airbnb Kaggle, garde les colonnes numériques utiles, nettoie les NaN."""
    df = pd.read_csv(chemin, encoding='latin-1', low_memory=False)

    colonnes = [
        "price",
        "minimum_nights",
        "accommodates",
        "review_scores_rating",
        "bedrooms",
    ]
    df = df[colonnes].copy()
    df = df.dropna()

    # supprimer les prix aberrants
    df = df[df["price"] > 0]
    df = df[df["price"] < 1000]

    print(f"Listings chargés : {df.shape[0]} lignes, {df.shape[1]} colonnes retenues")
    print(df.describe().round(1))
    return df

In [16]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette."""
    print(f"{'k':<5} {'inertie':>10} {'silhouette':>12}")
    print("-" * 30)

    meilleur_k, meilleur_sil = 2, -1
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        km.fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_)
        print(f"{k:<5} {km.inertia_:>10.1f} {sil:>12.3f}")
        if sil > meilleur_sil:
            meilleur_sil, meilleur_k = sil, k

    print(f"\nSegment retenu : k={meilleur_k}  (silhouette={meilleur_sil:.3f})")
    return meilleur_k

In [ ]:

CHEMIN = "Listings.csv"

colonnes_features = ["price", "minimum_nights", "accommodates",
                     "review_scores_rating", "bedrooms"]

# === CAS NORMAL ===
print("=== CAS NORMAL ===\n")
df_airbnb = charger_airbnb(CHEMIN)

# KMeans sur un échantillon de 20 000 lignes (279k = trop lent sinon)
df_sample = df_airbnb.sample(n=20_000, random_state=42)

scaler_ab = StandardScaler()
X_scaled  = scaler_ab.fit_transform(df_sample[colonnes_features])

k_optimal = choisir_k(X_scaled)

km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_sample = df_sample.copy()
df_sample["segment"] = km_final.fit_predict(X_scaled)

print(f"\n=== DESCRIPTION DES {k_optimal} SEGMENTS ===\n")
desc = df_sample.groupby("segment")[colonnes_features].mean().round(1)
print(desc)

print("\n--- Interprétation ---")
for seg in range(k_optimal):
    prix   = desc.loc[seg, "price"]
    nuits  = desc.loc[seg, "minimum_nights"]
    cap    = desc.loc[seg, "accommodates"]
    note   = desc.loc[seg, "review_scores_rating"]
    if prix > desc["price"].median() * 1.8:
        label = "Premium (prix élevé)"
    elif nuits > 20:
        label = "Longue durée (séjours longs)"
    elif cap >= 4:
        label = "Familial / groupe"
    elif note >= 95:
        label = "Très bien noté / qualité"
    else:
        label = "Budget / standard"
    print(f"  Segment {seg} → {label}  "
          f"(prix={prix:.0f}€, min_nuits={nuits:.0f}, capacité={cap:.0f}, note={note:.0f})")

# === CAS LIMITE : sans standardiser ===
print("\n=== CAS LIMITE (sans standardisation) ===\n")
X_brut = df_sample[colonnes_features].values
km_brut = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
labels_brut = km_brut.fit_predict(X_brut)
sil_brut   = silhouette_score(X_brut, labels_brut)
sil_scaled = silhouette_score(X_scaled, km_final.labels_)
print(f"Silhouette SANS standardisation : {sil_brut:.3f}")
print(f"Silhouette AVEC standardisation : {sil_scaled:.3f}")
print("→ 'price' (0–1000) écrase 'bedrooms' (1–5) sans standardisation.")
print("  Les segments n'ont plus de sens métier.")

# === CAS ADVERSARIAL : annonce à 100 000€/nuit ===
print("\n=== CAS ADVERSARIAL (outlier prix=100 000) ===\n")
df_poison = df_sample[colonnes_features].copy()
outlier = pd.DataFrame([[100_000, 1, 2, 100, 1]], columns=colonnes_features)
df_poison = pd.concat([df_poison, outlier], ignore_index=True)

scaler_p = StandardScaler()
X_poison = scaler_p.fit_transform(df_poison)
km_poison = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_poison.fit(X_poison)

print("Centre 'price' AVANT outlier par segment :")
for i, c in enumerate(km_final.cluster_centers_):
    print(f"  Segment {i} : price_scaled={c[0]:.3f}")
print("\nCentre 'price' APRÈS outlier par segment :")
for i, c in enumerate(km_poison.cluster_centers_):
    print(f"  Segment {i} : price_scaled={c[0]:.3f}")
print("\n→ L'outlier déforme les centres. Nettoyage J2 obligatoire avant tout KMeans.")

=== CAS NORMAL ===



UnicodeDecodeError: 'utf-8' codec can't decode byte 0x81 in position 206899: invalid start byte